In [1]:
%pip install feedparser transformers torch numpy

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for sgmllib3k: filename=sgmllib3k-1.0.0-py3-none-any.whl size=6104 sha256=c905e8cf4adfea761c2ae80b08ad963ae97d500dbb480d61e2ee8bd86b8f7708
  Stored in directory: c:\users\atharv jadhav\appdata\local\pip\cache\wheels\f0\69\93\a47e9d621be168e9e33c7ce60524393c0b92ae83cf6c6e89c5
Successfully built sgmllib3k

   -------------------- ------------------- 1/2 [feedparser]
   -------------------- ------------------- 1/2 [feedparser]
   -------------------- ------------------- 1/2 [feedparser]
   -------------------- ------------------- 1/2 [feedparser]
   -------------------- ------------------- 1/2 [feedparser]
   -------------------- -------------------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import feedparser
import urllib.parse

def fetch_news(commodity, max_articles=15):

    query = f"{commodity} India agriculture price mandi"
    encoded_query = urllib.parse.quote(query)

    url = (
        f"https://news.google.com/rss/search?"
        f"q={encoded_query}&hl=en-IN&gl=IN&ceid=IN:en"
    )

    feed = feedparser.parse(url)

    articles = []

    for entry in feed.entries[:max_articles]:
        articles.append({
            "title": entry.title,
            "summary": entry.summary if hasattr(entry, "summary") else ""
        })

    return articles


In [8]:
articles = fetch_news("Tomato")
len(articles)
articles[:3]


[{'title': 'Centre launches subsidised tomato sale at Rs 47-60/kg in Delhi as prices spike due to rains - The Economic Times',
  'summary': '<a href="https://news.google.com/rss/articles/CBMi_wFBVV95cUxPRjM4ekd4RlVub2NsRm9GR0ZFUVBic3RyNWpQLWp6MDYtTHNCLUtjVEMtQjBnYlN3WUpla0VMcjhjbWV4TUhxWkN0N0RwRi1DS3o5MmRGRjBTaGZqS0F5TXVfWURQV0NaOXZkUDQyR2hwRUtCSkFibzNxV2xYcEFDZm9QeVR1bDF4UzVtcThiaXBScDNHdndKUU5oLW5Ea0M4TEVvZUlzaTlwT1FmSkdYUE1vYUE5WlY2UVdRelVGZVI4Y21QNTRkdTY2U3JHUkRCb0ZqajVDTi1EVTF3eEstb2RsM2xlaHZreWlIMnVhQjh3ZHFESkxUU0tfaVZsMDTSAYQCQVVfeXFMT2dwcHAwdDlCa3NDbm9QOWpNMElXdTZ1UHllblN1Z21mVTZoZ3NoWHB3dHZNaUtlcmF1ZHB6QnA4eDkxaDdfY3Vwc2RiU2xhbzVFLThMbGJZM0lvTU1TejlINFJtQWt1SnphanFqMldETm1jc3FTYmNuMjN3Nkk1eXVQV3FYeEt3c0Fra2U3ZmpMTnVrOFFTSUpkR1BaV2RTSWdOTXNIQ19nTDcxZUY1djJTbHBmVHdTR1VhUDlyUHhmdlpNZVdkSzlDMWl4b1NlNFhuYmc0aXRlWkJDMWVTLWJzYnV0WnRkNUNhZHZLWC15WVRNM3hjOW5qbGwweEZyTXpQX2w?oc=5" target="_blank">Centre launches subsidised tomato sale at Rs 47-60/kg in Delhi as prices spike due to rains

In [9]:
from transformers import pipeline
import numpy as np

# Load once (important)
sentiment_model = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english"
)

def compute_sentiment(articles):
    if len(articles) == 0:
        return 0, 0  # neutral fallback

    scores = []
    
    for art in articles:
        text = art["title"]
        result = sentiment_model(text)[0]
        
        score = result["score"]
        if result["label"] == "NEGATIVE":
            score = -score
        
        scores.append(score)

    avg_sentiment = np.mean(scores)
    return avg_sentiment, len(scores)


c:\Users\Atharv Jadhav\ML PROJECTS\COMPLETE-SUPPY-CHAIN\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 104/104 [00:00<00:00, 366.88it/s, Materializing param=pre_classifier.weight]                                  


In [10]:
avg_sentiment, news_count = compute_sentiment(articles)

print("Average Sentiment:", avg_sentiment)
print("News Count:", news_count)


Average Sentiment: -0.59436407883962
News Count: 15


In [11]:
def compute_risk_index(sentiment, news_count):
    if news_count == 0:
        return 0
    
    intensity = abs(sentiment)
    risk = intensity * news_count
    return risk


In [12]:
risk = compute_risk_index(avg_sentiment, news_count)
print("Risk Index:", risk)


Risk Index: 8.9154611825943


In [14]:
def adjust_price_with_risk(base_price, sentiment, risk_index):
    
    # Small calibrated adjustment (keep realistic)
    adjustment_factor = 0
    
    if risk_index > 5:
        if sentiment < 0:
            # Negative news → possible price spike
            adjustment_factor = 0.02   # +2%
        else:
            # Positive news → possible drop
            adjustment_factor = -0.015  # -1.5%
    
    adjusted_price = base_price * (1 + adjustment_factor)
    
    return adjusted_price


In [15]:
base_price = 9500  # example from fusion model

adjusted_price = adjust_price_with_risk(
    base_price,
    avg_sentiment,
    risk
)

print("Base Forecast:", base_price)
print("Risk Adjusted Forecast:", adjusted_price)


Base Forecast: 9500
Risk Adjusted Forecast: 9690.0


In [16]:
def safety_stock_recommendation(
    demand_avg,
    demand_std,
    lead_time_days,
    risk_index
):
    z_score = 1.65  # 95% service level
    
    base_safety_stock = z_score * demand_std * np.sqrt(lead_time_days)
    
    # Increase buffer if risk high
    risk_multiplier = 1 + min(risk_index / 20, 0.5)
    
    adjusted_safety_stock = base_safety_stock * risk_multiplier
    
    return adjusted_safety_stock



In [17]:
safety_stock = safety_stock_recommendation(
    demand_avg=100,
    demand_std=20,
    lead_time_days=7,
    risk_index=risk
)

print("Recommended Safety Stock:", safety_stock)


Recommended Safety Stock: 126.23014690091213


In [18]:
def generate_reasoning(commodity, sentiment, news_count, risk_index):
    
    if news_count == 0:
        return f"No significant recent news detected for {commodity}. Market appears stable."
    
    if sentiment < -0.3:
        return (
            f"Recent negative news coverage ({news_count} articles) "
            f"suggests potential supply stress or price volatility "
            f"for {commodity}. Risk index: {risk_index:.2f}."
        )
    
    if sentiment > 0.3:
        return (
            f"Positive market signals detected for {commodity}. "
            f"Supply conditions may improve. Risk index: {risk_index:.2f}."
        )
    
    return (
        f"Mixed news environment for {commodity}. "
        f"Monitor closely. Risk index: {risk_index:.2f}."
    )


In [19]:
explanation = generate_reasoning(
    "Tomato",
    avg_sentiment,
    news_count,
    risk
)

print(explanation)


Recent negative news coverage (15 articles) suggests potential supply stress or price volatility for Tomato. Risk index: 8.92.
